# Exercise 10: SciPy

The weather of 2023 at the KMI stations: daily at 13 stations in `../../data/raw/kmi_daily_2023.csv`, and hourly
at five stations in `../../kmi/kmi_hourly_2023.parquet`.

In [1]:
import numpy as np
import pandas as pd


def annual_cycle(day, mean, amplitude, peak):
    angle = 2 * np.pi * (day - peak) / 365
    return mean + amplitude * np.cos(angle)

**10.1** Fit the yearly cycle of the daily mean temperature for ZEEBRUGGE and for MONT RIGI. Compare the
mean, the amplitude and the peak. What does the sea do?

Docs: [`curve_fit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)

In [2]:
from scipy import optimize

daily = pd.read_csv("../../data/raw/kmi_daily_2023.csv", parse_dates=["timestamp"])
for name in ["ZEEBRUGGE", "MONT RIGI"]:
    station = daily[daily["name"] == name]
    params, _ = optimize.curve_fit(
        annual_cycle, station["timestamp"].dt.day_of_year, station["temp_avg"], p0=[10, 8, 200]
    )
    print(name, params)

ZEEBRUGGE [ 12.52219178   6.71313366 216.07926155]
MONT RIGI [  8.37460274   7.9674265  207.79865591]


Mont Rigi is about 4 °C colder all year. Zeebrugge has the smallest amplitude, and its peak comes about a week later: the sea warms up and cools down slowly.

## Bonus

**Bonus 1** The temperature also has a daily cycle: cool at night, warmest in the afternoon. Write a function
`daily_cycle`, like `annual_cycle`, but for a cycle of 24 hours.

In [3]:
def daily_cycle(hour, mean, amplitude, peak):
    angle = 2 * np.pi * (hour - peak) / 24
    return mean + amplitude * np.cos(angle)

**Bonus 2** Read the hourly observations, `../../kmi/kmi_hourly_2023.parquet`. The temperature is in
`temp_dry_shelter_avg`, and the station is a code: ZEEBRUGGE, on the coast, is 6418, and MELLE, inland, is
6434. Fit the daily cycle for both stations, and compare the amplitudes.

In [4]:
hourly = pd.read_parquet("../../kmi/kmi_hourly_2023.parquet")
for code in [6418, 6434]:
    station = hourly[hourly["code"] == code]
    params, _ = optimize.curve_fit(
        daily_cycle, station["timestamp"].dt.hour, station["temp_dry_shelter_avg"], p0=[10, 3, 14]
    )
    print(code, params)

6418 [12.52281964  1.10390154 15.52081823]
6434 [12.18185959  2.69775598 14.76842779]


Zeebrugge goes 1.1 °C up and down around its mean every day, Melle 2.7 °C: the sea also dampens the daily cycle.

**Bonus 3** At what time of the day is it warmest? The times are in UTC.

In the afternoon. The fitted peak is at 14.8 hours for Melle and 15.5 hours for Zeebrugge. But each hourly value is the mean of the hour before its time (the value at 15:00 covers 14:00 to 15:00), so the real peak is half an hour earlier: around 14:15 UTC in Melle and 15:00 UTC in Zeebrugge. In Belgian time, add one hour in winter and two in summer.